# Bermuda Python Environment

## Environment overview
This is a Python environment you can use for data analysis on the Oxford Earth Sciences Bermuda field course. The following packages are available:

* `numpy`
* `scipy`
* `matplotlib`
* `pandas`
* `gsw`
* `PyCO2SYS`

The python file `bermuda_functions.py` contains the following useful helper functions for importing data. 

* `read_oxford_ctd(fh)`: reads data from raw Valeport miniCTD text file output and produces a Pandas dataframe with pressure, temperature, and salinity. Takes the file handle as an input and returns the data and metadata (as a dictionary).
* `read_bios_ctd(fh)`: reads data from BIOS CTD CSV files and produces two dictionaries with cast IDs as keys. The first contains a formatted Pandas dataframe for that cast, and the second contains the dates for each cast. Takes the file handle as an input.

## Datasets
Here is an overview of the directory structure so you can obtain datasets you might need:
```
data/
    bats/
        859990_v8_hydrostation_s_bottle.csv  (Hydrostation S bottle data for SL practical)
    ctd/
        example_dataset.TXT                  (Ignore - example dataset for testing)
        ...                                  (We will put the output from the miniCTD here)
    oa/
        CTD_Full_Transect_OA_70125OS.csv     (CTD data from all OA stations, Cruise 70125 September 2025)
        CTD_Full_Transect_OA_70131OS.csv     (CTD data from all OA stations, Cruise 70131 March 2026)
        CTD_timeseries_OA_OS_Station_1.csv   (CTD data from OS-1 for cruises 70020-70135 [2016-2026])
        CTD_timeseries_OA_OS_Station_2.csv   (CTD data from OS-2 for cruises 70020-70135 [2016-2026])
        CTD_timeseries_OA_OS_Station_3.csv   (CTD data from OS-3 for cruises 70020-70135 [2016-2026])
        OA_OS_bottle_data_Stations_1_2_3.csv (Bottle data from OA-OS stations 1-3 for depths shallower than 11 m [2014-2026])
    tg/
        h259_pre1985.csv                     (Pre-1985 tide gauge data for the SL practical)
        h259.csv                             (Post-1985 tide gauge data for the SL practical)
```

In [ ]:
# Now you can write your own code...
# Example 1 (importing and plotting our miniCTD data):

import matplotlib.pyplot as plt
import numpy as np
from bermuda_functions import read_ctd

data, metadata = read_ctd('data/ctd/example_dataset.TXT') # Get CTD data
data = data[data['P'] > 0.1] # Remove data with P < 0.1 dbar (surface)
smin, smax = np.quantile(data['S'], 0.1), np.quantile(data['S'], 0.9)

# Create a simple plot
f, ax = plt.subplots(1, 1, figsize=(4, 6))
plot = ax.scatter(data['T'], data['P'], c=data['S'], marker='.', s=1,
                  vmin=smin, vmax=smax) # Color points by salinity
ax.yaxis.set_inverted(True)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('Temperature (C)')
ax.set_ylabel('Pressure (dbar)')
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.colorbar(plot, label='Salinity (PSU)')


In [ ]:
# Example 2 (importing and plotting dissolved oxygen and fluorescence data from Station OS-3):

import matplotlib.pyplot as plt
import numpy as np
from bermuda_functions import read_bios_ctd

months = [3] # Only plot data for particular months

data, times = read_bios_ctd('data/oa/CTD_timeseries_OA_OS_Station_3.csv') # Import data
cast_ids = list(data.keys())

# Create a simple plot for all dissolved oxygen profiles
f, ax = plt.subplots(1, 2, figsize=(8, 6))

for cast_id in cast_ids:
    if times[cast_id].month in months:
        plot = ax[0].plot(data[cast_id]['dissolved_oxygen_umol_kg-1'], data[cast_id]['pressure_dbar'], c='b', linewidth=1)
        plot = ax[1].plot(data[cast_id]['fluorescence_mg_m-3'], data[cast_id]['pressure_dbar'], c='g', linewidth=1)

for axis in ax:
    axis.yaxis.set_inverted(True)
    axis.xaxis.tick_top()
    axis.xaxis.set_label_position('top')
    axis.spines['right'].set_visible(False)
    axis.spines['bottom'].set_visible(False)
    
ax[0].set_ylabel('Pressure (dbar)')
ax[0].set_xlabel(r'Dissolved oxygen ($\mu$mol kg$^{-1}$)')
ax[1].set_xlabel(r'Chlorophyll-a (RFU)')